# 🐼 pandas: sua primeira tabela de verdade

O gestor de cartões chega com um arquivo de **30 mil clientes** e faz três perguntas:

1. *"Quantos clientes temos e quais informações a base traz?"*
2. *"Qual é o limite médio e a idade média dos clientes?"*
3. *"Me mostre só os clientes com mais de 60 anos e limite acima de 200 mil."*

No Excel, 30 mil linhas já pesam, e cada pergunta vira filtro, fórmula e tabela dinâmica. Com o **pandas**, cada pergunta é uma linha de código, e você pode repetir a análise amanhã, com a base atualizada, sem refazer nada à mão.

Na aula passada vimos o NumPy, que é ótimo para números. Hoje vamos ver o que fazer quando a tabela mistura **números, textos e categorias**: é o pandas.

## 🗺️ Roteiro

- 🔁 Por que o NumPy não basta para uma tabela real
- 🧮 `Series` e `DataFrame`: os dois objetos do pandas
- 🏦 Conhecer a base de cartão de crédito (e o Kaggle)
- 👀 Olhar uma tabela: `head`, `shape`, `info`, `describe`
- 🎯 Selecionar colunas e linhas: `[]`, `loc`, `iloc`
- 🔎 Filtrar com condições: `&`, `|`, `isin`

**No fim da aula você vai saber:**
1. criar e olhar um `DataFrame`;
2. baixar uma base do Kaggle e carregá-la com `pd.read_csv`;
3. selecionar colunas e linhas específicas;
4. filtrar clientes por uma ou mais condições;
5. responder perguntas de negócio sem abrir o Excel.

## ⚙️ Configuração

Vou importar as bibliotecas de sempre e o `kagglehub`, que baixa bases do Kaggle. No Colab ele costuma já estar instalado, mas a primeira linha garante.

In [1]:
# Esta célula instala o kagglehub, caso ainda não esteja disponível.
!pip install -q kagglehub


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Esta célula importa as bibliotecas e mostra as versões.
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

NumPy: 2.5.4
pandas: 3.0.6


## 🔁 Revisão: onde o NumPy trava

O `ndarray` exige que **todos os elementos tenham o mesmo tipo**. Mas uma tabela de clientes tem nome (texto), idade (inteiro) e limite (decimal). Vou colocar tudo num array e ver o que acontece.

In [3]:
# Esta célula monta um array com nome, idade e limite de três clientes.
tabela = np.array([
    ["Ana", 35, 12000.0],
    ["Bruno", 52, 8500.0],
    ["Carla", 28, 20000.0],
])
print(tabela)
print(tabela.dtype)

[['Ana' '35' '12000.0']
 ['Bruno' '52' '8500.0']
 ['Carla' '28' '20000.0']]
<U32


Repare nas aspas: o NumPy transformou **tudo em texto** (`<U32` quer dizer "texto de até 32 caracteres"). Para ele, a idade `35` virou `'35'`. Vamos tentar calcular o limite médio. **Vou provocar o erro de propósito:**

In [4]:
# Esta célula tenta calcular a média da coluna de limites, que virou texto.
tabela[:, 2].mean()

TypeError: the resolved dtypes are not compatible with add.reduce. Resolved (dtype('<U32'), dtype('<U32'), dtype('<U64'))

**Como ler:** `UFuncTypeError: the resolved dtypes are not compatible with add.reduce`. Traduzindo: "os tipos que encontrei (texto) não servem para **somar**". O NumPy é ótimo para tabelas só de números, mas uma tabela real mistura tipos. É aí que entra o pandas: **cada coluna tem o seu próprio tipo**.

## 🧮 Conhecendo o pandas

O pandas tem dois objetos principais:

| Objeto | O que é | Analogia |
|---|---|---|
| **`Series`** | Uma **coluna**: uma lista de valores com um rótulo para cada posição | Uma coluna do Excel, com o número da linha ao lado |
| **`DataFrame`** | Uma **tabela**: várias `Series` lado a lado, cada uma com seu tipo | A planilha inteira |

### A `Series`

Vou criar uma coluna com o limite de crédito de três clientes.

In [5]:
# Esta célula cria uma Series com o limite de crédito de três clientes.
limites = pd.Series([12000.0, 8500.0, 20000.0])
limites

0    12000.0
1     8500.0
2    20000.0
dtype: float64

À esquerda estão os **rótulos** (`0`, `1`, `2`), chamados de **índice**. À direita, os valores. No fim aparece o tipo da coluna: `float64`, decimal. As contas funcionam como no NumPy:

In [6]:
# Esta célula calcula a média e aplica um aumento de 10% em todos os limites.
print("Média:", limites.mean())
print(limites * 1.10)

Média: 13500.0
0    13200.0
1     9350.0
2    22000.0
dtype: float64


Conferindo na mão: (12000 + 8500 + 20000) ÷ 3 = **13.500**. E 12.000 × 1,10 = **13.200**. O aumento foi aplicado a todos de uma vez, sem laço.

### O `DataFrame`

Agora a tabela completa: junto várias colunas, **cada uma com o seu tipo**. O jeito mais simples é um dicionário, em que a chave é o nome da coluna.

In [7]:
# Esta célula cria um DataFrame com nome, idade e limite de quatro clientes.
clientes_exemplo = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carla", "Diego"],
    "idade": [35, 52, 28, 61],
    "limite": [12000.0, 8500.0, 20000.0, 5000.0],
})
clientes_exemplo

,nome,idade,limite
0,Ana,35,12000.0
1,Bruno,52,8500.0
2,Carla,28,20000.0
3,Diego,61,5000.0


In [8]:
# Esta célula mostra o tipo de cada coluna do DataFrame.
clientes_exemplo.dtypes

nome          str
idade       int64
limite    float64
dtype: object

Agora cada coluna tem o **seu** tipo: `idade` é inteiro, `limite` é decimal e `nome` é texto. Por isso o pandas faz conta com a coluna certa sem reclamar. 💡 Dependendo da versão do pandas, o texto aparece como `str` ou como `object`; para nós é a mesma coisa. Vamos ver a média do limite, que no NumPy deu erro:

In [9]:
# Esta célula calcula a média do limite no DataFrame de exemplo.
clientes_exemplo["limite"].mean()

np.float64(11375.0)

Conferindo: (12000 + 8500 + 20000 + 5000) ÷ 4 = **11.375**. Funcionou, porque a coluna `limite` é numérica.

## 🏦 A base que vamos usar

Até aqui foram quatro clientes inventados. Agora vem a base de verdade.

### O que é o Kaggle

O **Kaggle** (kaggle.com) é um site com **milhares de bases de dados públicas**, além de competições e notebooks de outras pessoas. É onde muita gente da área de dados treina e busca dados para estudar. Para **baixar** as bases públicas pelo código, como faremos, não precisa nem de conta. Mas vale criar uma, é gratuita e útil para explorar.

### A base: *Default of Credit Card Clients*

- **De onde vem:** coletada pela universidade de Taiwan e publicada no repositório da **UCI** (um repositório acadêmico), e disponível no Kaggle em `uciml/default-of-credit-card-clients-dataset`.
- **O que tem:** **30 mil clientes de cartão de crédito**, de abril a setembro de 2005, com perfil, histórico de pagamentos e se ficaram inadimplentes no mês seguinte.
- ⚠️ **Cuidado:** os valores estão em **NT$** (dólar taiwanês), não em reais. O que importa para a aula é a estrutura, que é parecida com a de qualquer base de cartão.

Os nomes originais das colunas estão em inglês e são pouco claros (`PAY_0`, `PAY_AMT3`). Vou renomear para o português, e este é o dicionário de dados **com os nomes já em português**:

| Coluna | O que é | Valores |
|---|---|---|
| `id_cliente` | Identificador do cliente | 1 a 30000 |
| `limite_credito` | Limite do cartão | NT$ |
| `sexo` | Sexo do cliente | 1 = masculino, 2 = feminino |
| `escolaridade` | Nível de escolaridade | 1 = pós-graduação, 2 = universidade, 3 = ensino médio, 4 = outros (5 e 6 = não informado) |
| `estado_civil` | Estado civil | 1 = casado, 2 = solteiro, 3 = outros |
| `idade` | Idade | anos |
| `atraso_set` ... `atraso_abr` | Situação do pagamento em cada mês (setembro até abril) | -1 = pago em dia; 1 = 1 mês de atraso; 2 = 2 meses; e assim por diante |
| `fatura_set` ... `fatura_abr` | Valor da fatura do mês | NT$ |
| `pago_set` ... `pago_abr` | Valor pago no mês | NT$ |
| `inadimplente` | Ficou inadimplente no mês seguinte? | 1 = sim, 0 = não |

💡 Nas colunas de atraso também aparecem os valores `-2` e `0` sem explicação oficial. Na prática, costumam ser lidos como "sem consumo" e "pagou o mínimo".

### Baixando e carregando

O `kagglehub` baixa a base e devolve a pasta onde o arquivo foi salvo.

In [10]:
# Esta célula baixa a base de cartão de crédito do Kaggle e mostra a pasta onde ela ficou.
caminho = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
caminho

'C:\\Users\\davih\\.cache\\kagglehub\\datasets\\uciml\\default-of-credit-card-clients-dataset\\versions\\1'

In [11]:
# Esta célula lê o arquivo CSV e guarda a tabela na variável clientes.
clientes = pd.read_csv(Path(caminho) / "UCI_Credit_Card.csv")
type(clientes)

pandas.DataFrame

O `pd.read_csv` lê um arquivo CSV (a "planilha de texto" que qualquer sistema exporta) e devolve um `DataFrame`. Agora vou renomear as colunas com um dicionário `"nome antigo": "nome novo"`.

In [12]:
# Esta célula renomeia as colunas da base para o português.
novos_nomes = {
    "ID": "id_cliente", "LIMIT_BAL": "limite_credito", "SEX": "sexo",
    "EDUCATION": "escolaridade", "MARRIAGE": "estado_civil", "AGE": "idade",
    "PAY_0": "atraso_set", "PAY_2": "atraso_ago", "PAY_3": "atraso_jul",
    "PAY_4": "atraso_jun", "PAY_5": "atraso_mai", "PAY_6": "atraso_abr",
    "BILL_AMT1": "fatura_set", "BILL_AMT2": "fatura_ago", "BILL_AMT3": "fatura_jul",
    "BILL_AMT4": "fatura_jun", "BILL_AMT5": "fatura_mai", "BILL_AMT6": "fatura_abr",
    "PAY_AMT1": "pago_set", "PAY_AMT2": "pago_ago", "PAY_AMT3": "pago_jul",
    "PAY_AMT4": "pago_jun", "PAY_AMT5": "pago_mai", "PAY_AMT6": "pago_abr",
    "default.payment.next.month": "inadimplente",
}
clientes = clientes.rename(columns=novos_nomes)
clientes.columns.tolist()

['id_cliente',
 'limite_credito',
 'sexo',
 'escolaridade',
 'estado_civil',
 'idade',
 'atraso_set',
 'atraso_ago',
 'atraso_jul',
 'atraso_jun',
 'atraso_mai',
 'atraso_abr',
 'fatura_set',
 'fatura_ago',
 'fatura_jul',
 'fatura_jun',
 'fatura_mai',
 'fatura_abr',
 'pago_set',
 'pago_ago',
 'pago_jul',
 'pago_jun',
 'pago_mai',
 'pago_abr',
 'inadimplente']

## 👀 Olhando a tabela

Antes de qualquer análise, a primeira coisa é **olhar** a base. Cinco comandos fazem isso.

### `head` e `shape`: as primeiras linhas e o tamanho

In [13]:
# Esta célula mostra as cinco primeiras linhas da tabela.
clientes.head()

,id_cliente,limite_credito,sexo,escolaridade,estado_civil,idade,atraso_set,atraso_ago,atraso_jul,atraso_jun,...,fatura_jun,fatura_mai,fatura_abr,pago_set,pago_ago,pago_jul,pago_jun,pago_mai,pago_abr,inadimplente
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [14]:
# Esta célula mostra o tamanho da tabela: linhas e colunas.
clientes.shape

(30000, 25)

São **30.000 linhas** (clientes) e **25 colunas**. O `shape` devolve sempre o par `(linhas, colunas)`.

### `info`: tipos e valores faltantes

In [15]:
# Esta célula mostra o tipo de cada coluna e quantos valores não nulos ela tem.
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_cliente      30000 non-null  int64  
 1   limite_credito  30000 non-null  float64
 2   sexo            30000 non-null  int64  
 3   escolaridade    30000 non-null  int64  
 4   estado_civil    30000 non-null  int64  
 5   idade           30000 non-null  int64  
 6   atraso_set      30000 non-null  int64  
 7   atraso_ago      30000 non-null  int64  
 8   atraso_jul      30000 non-null  int64  
 9   atraso_jun      30000 non-null  int64  
 10  atraso_mai      30000 non-null  int64  
 11  atraso_abr      30000 non-null  int64  
 12  fatura_set      30000 non-null  float64
 13  fatura_ago      30000 non-null  float64
 14  fatura_jul      30000 non-null  float64
 15  fatura_jun      30000 non-null  float64
 16  fatura_mai      30000 non-null  float64
 17  fatura_abr      30000 non-null  float64
 1

Todas as colunas têm **30.000 valores não nulos**, ou seja, **não há valores faltantes** nesta base. Os tipos são `int64` (inteiros) e `float64` (decimais).

### `describe`: o resumo estatístico

Mostra, para cada coluna numérica, contagem, média, desvio, mínimo, quartis e máximo. A base tem muitas colunas, então vou olhar só as duas que importam agora.

In [16]:
# Esta célula mostra o resumo estatístico das colunas idade e limite_credito.
clientes[["idade", "limite_credito"]].describe()

,idade,limite_credito
count,30000.000000,30000.000000
mean,35.485500,167484.322667
std,9.217904,129747.661567
min,21.000000,10000.000000
25%,28.000000,50000.000000
50%,34.000000,140000.000000
75%,41.000000,240000.000000
max,79.000000,1000000.000000


A **idade média** é de **35,5 anos**, com mínimo de 21 e máximo de 79. O **limite médio** é de **NT$ 167.484**, mas o máximo chega a NT$ 1.000.000: alguns clientes têm um limite muito acima da média.

💡 Usei `[["idade", "limite_credito"]]` para escolher duas colunas. Na próxima parte vamos ver com calma como a seleção funciona.

### Uma primeira resposta para o gestor

E a taxa de inadimplência? A coluna `inadimplente` tem `1` para quem não pagou e `0` para quem pagou. A **média** de uma coluna de 0 e 1 é a **proporção de uns**.

In [17]:
# Esta célula calcula a proporção de clientes inadimplentes.
clientes["inadimplente"].mean()

np.float64(0.2212)

Cerca de **22%** dos clientes ficaram inadimplentes: 6.636 de 30.000, quase 1 em cada 5. Já temos uma resposta de negócio com **uma linha de código**.

## 🎯 Seleção: pegando só o que interessa

Quase toda pergunta de negócio começa por **escolher um pedaço da tabela**: uma coluna, algumas colunas, uma linha. Vou mostrar cada jeito em pequeno, com a tabela de quatro clientes, antes de usar na base real.

Para a visita, vou usar o **nome** como rótulo de cada linha.

In [18]:
# Esta célula cria a tabela de exemplo usando o nome do cliente como rótulo das linhas.
pessoas = clientes_exemplo.set_index("nome")
pessoas

,idade,limite
nome,,
Ana,35,12000.0
Bruno,52,8500.0
Carla,28,20000.0
Diego,61,5000.0


### 1. Uma coluna: colchetes simples

Devolve uma `Series`.

In [19]:
# Esta célula seleciona a coluna idade.
pessoas["idade"]

nome
Ana      35
Bruno    52
Carla    28
Diego    61
Name: idade, dtype: int64

### 2. Várias colunas: colchetes duplos

A lista de nomes vai **dentro de outro colchete**. Devolve um `DataFrame`.

In [20]:
# Esta célula seleciona as colunas idade e limite.
pessoas[["idade", "limite"]]

,idade,limite
nome,,
Ana,35,12000.0
Bruno,52,8500.0
Carla,28,20000.0
Diego,61,5000.0


Fácil de esquecer: os colchetes **duplos** são "uma lista de colunas dentro do colchete". Se escrever `pessoas["idade", "limite"]`, dá erro.

### 3. Uma linha pelo **rótulo**: `loc`

O `loc` (de *location*) escolhe pelo **nome** da linha e da coluna, no formato `loc[linha, coluna]`.

In [21]:
# Esta célula seleciona o limite da Carla, pelo rótulo da linha e o nome da coluna.
pessoas.loc["Carla", "limite"]

np.float64(20000.0)

In [22]:
# Esta célula seleciona todas as informações da Carla (a linha inteira).
pessoas.loc["Carla"]

idade        28.0
limite    20000.0
Name: Carla, dtype: float64

### 4. Uma linha pela **posição**: `iloc`

O `iloc` (o *i* é de *integer*) escolhe pela **posição**, contando a partir do zero, como nas listas.

In [23]:
# Esta célula seleciona a primeira linha e a segunda coluna, pela posição.
pessoas.iloc[0, 1]

np.float64(12000.0)

Posição `0` é a primeira linha (Ana) e posição `1` é a segunda coluna (`limite`): o resultado é **12.000**. Resumindo: `loc` usa **nomes**, `iloc` usa **números**.

### 5. Fatias: uma diferença que pega todo mundo

Para pegar um intervalo, usamos `:`. Repare que o `loc` **inclui o último**, e o `iloc` **não inclui**, como nas listas.

In [24]:
# Esta célula pega de Ana até Carla com loc (inclui Carla) e as duas primeiras linhas com iloc.
print(pessoas.loc["Ana":"Carla", "idade"])
print()
print(pessoas.iloc[0:2, 0])

nome
Ana      35
Bruno    52
Carla    28
Name: idade, dtype: int64

nome
Ana      35
Bruno    52
Name: idade, dtype: int64


O `loc["Ana":"Carla"]` trouxe **3 clientes**, incluindo Carla, e o `iloc[0:2]` trouxe **2**. Guarde essa regra: o `loc` fatia pelo nome e inclui o fim.

Dá para fatiar colunas também. Para pegar de `idade` até `limite`: `pessoas.loc[:, "idade":"limite"]`, onde o `:` sozinho significa "todas as linhas".

### 6. Criar uma coluna nova

Basta atribuir a um nome que ainda não existe. Vou calcular o limite com um aumento de 20%.

In [25]:
# Esta célula cria a coluna limite_novo com um aumento de 20% sobre o limite.
pessoas["limite_novo"] = pessoas["limite"] * 1.20
pessoas

,idade,limite,limite_novo
nome,,,
Ana,35,12000.0,14400.0
Bruno,52,8500.0,10200.0
Carla,28,20000.0,24000.0
Diego,61,5000.0,6000.0


Conferindo: 12.000 × 1,20 = **14.400**, o valor da Ana. A conta foi feita para todas as linhas de uma vez.

### 7. Ordenar: `sort_values`

In [26]:
# Esta célula ordena os clientes do maior para o menor limite.
pessoas.sort_values("limite", ascending=False)

,idade,limite,limite_novo
nome,,,
Carla,28,20000.0,24000.0
Ana,35,12000.0,14400.0
Bruno,52,8500.0,10200.0
Diego,61,5000.0,6000.0


Carla (20.000) vem primeiro. Por padrão a ordem é crescente, por isso usei `ascending=False`.

### Um erro comum: errar o nome da coluna

**Vou provocar o erro de propósito:** `Idade` com letra maiúscula, em vez de `idade`.

In [27]:
# Esta célula tenta selecionar uma coluna com o nome escrito errado.
pessoas["Idade"]

KeyError: 'Idade'

**Como ler:** `KeyError: 'Idade'`. Traduzindo: "não existe nenhuma chave (coluna) com esse nome". Letra maiúscula, acento ou espaço fazem diferença. Na dúvida, veja os nomes com `pessoas.columns`.

💡 **Um cuidado:** para pegar um valor, use `loc[linha, coluna]` e **não** `pessoas["limite"]["Carla"]`. O segundo funciona hoje, mas é frágil e pode dar problemas ao tentar alterar o valor.

## 🛠️ Casos resolvidos: seleção

Agora a base real. Cada pergunta é resolvida duas vezes: primeiro do jeito que você já conhece, depois com o pandas.

### 💰 Caso 1: Financeiro, conhecendo o cliente

O analista de risco quer olhar clientes específicos.

**Pergunta 1: quais são os limites dos cinco primeiros clientes?**

*Com lista:* percorrer a coluna e guardar os cinco primeiros.

In [28]:
# Esta célula pega os cinco primeiros limites usando uma lista e um laço.
primeiros_limites = []
for limite in clientes["limite_credito"]:
    if len(primeiros_limites) < 5:
        primeiros_limites.append(limite)
primeiros_limites

[20000.0, 120000.0, 90000.0, 50000.0, 50000.0]

*Com pandas:* uma linha, pelo rótulo (de 0 a 4, inclusive).

In [29]:
# Esta célula pega os cinco primeiros limites com loc.
clientes.loc[0:4, "limite_credito"]

0     20000.0
1    120000.0
2     90000.0
3     50000.0
4     50000.0
Name: limite_credito, dtype: float64

Os cinco limites são **20.000, 120.000, 90.000, 50.000 e 50.000**. Repare que o `loc[0:4]` incluiu o 4, e por isso vieram cinco clientes.

**Pergunta 2: como é o perfil do cliente de `id_cliente` 100?**

Os rótulos das linhas começam em 0, então o cliente 100 está na linha de rótulo 99. Vou ver só algumas colunas dele.

In [30]:
# Esta célula mostra o perfil do cliente 100 (linha de rótulo 99), em algumas colunas.
clientes.loc[99, ["id_cliente", "idade", "limite_credito", "fatura_set", "inadimplente"]]

id_cliente          100.0
idade                38.0
limite_credito    20000.0
fatura_set        17973.0
inadimplente          1.0
Name: 99, dtype: float64

É um cliente de **38 anos**, com limite de **20.000**, fatura de setembro de **17.973** e que **ficou inadimplente** (`inadimplente = 1`). A fatura já está perto do limite, um sinal de atenção.

**Pergunta 3: como montar uma visão enxuta, só com as colunas que o gestor pediu?**

*Com pandas:* uma lista de colunas e guardamos o resultado numa variável.

In [31]:
# Esta célula monta uma tabela enxuta com quatro colunas e mostra o tamanho e as primeiras linhas.
visao_enxuta = clientes[["id_cliente", "idade", "limite_credito", "inadimplente"]]
print(visao_enxuta.shape)
visao_enxuta.head()

(30000, 4)


,id_cliente,idade,limite_credito,inadimplente
0,1,24,20000.0,1
1,2,26,120000.0,1
2,3,34,90000.0,0
3,4,37,50000.0,0
4,5,57,50000.0,0


A tabela passou de **25 para 4 colunas**, com as mesmas 30.000 linhas. É assim que se prepara uma planilha para enviar ao gestor sem expor colunas desnecessárias.

**Pergunta 4: qual o total faturado do cliente 1 nos seis meses?**

As colunas de fatura estão lado a lado, de `fatura_set` a `fatura_abr`. Uma **fatia de colunas** pega todas de uma vez.

In [32]:
# Esta célula pega as seis faturas do primeiro cliente e soma o total faturado.
faturas_cliente_1 = clientes.loc[0, "fatura_set":"fatura_abr"]
print(faturas_cliente_1)
print("Total:", faturas_cliente_1.sum())

fatura_set    3913.0
fatura_ago    3102.0
fatura_jul     689.0
fatura_jun       0.0
fatura_mai       0.0
fatura_abr       0.0
Name: 0, dtype: float64
Total: 7704.0


Conferindo na mão: 3.913 + 3.102 + 689 + 0 + 0 + 0 = **7.704**. Lendo de abril (à direita) para setembro (à esquerda), a fatura desse cliente começou zerada e foi crescendo até **3.913**.

✅ **O que o caso mostrou:** `loc` para linhas e colunas pelo **nome**; colchetes duplos para escolher colunas; fatias para pegar um bloco de colunas.

### 🛒 Caso 2: Comercial, campanha de aumento de limite

A equipe comercial quer oferecer aumento de limite para quem **usa bastante** o cartão. Para isso precisa de uma medida nova: o **uso do limite**, que é a fatura dividida pelo limite.

**Pergunta 1: qual o uso do limite de cada cliente?**

*Com lista:* um laço que divide cada fatura pelo limite.

In [33]:
# Esta célula calcula o uso do limite com um laço, cliente por cliente.
uso_lista = []
for fatura, limite in zip(clientes["fatura_set"], clientes["limite_credito"]):
    uso_lista.append(fatura / limite)
len(uso_lista)

30000

*Com pandas:* dividir uma coluna pela outra e guardar numa coluna nova.

In [34]:
# Esta célula cria a coluna uso_limite dividindo a fatura de setembro pelo limite.
clientes["uso_limite"] = clientes["fatura_set"] / clientes["limite_credito"]
clientes["uso_limite"].describe()

count    30000.000000
mean         0.423771
std          0.411462
min         -0.619892
25%          0.022032
50%          0.313994
75%          0.829843
max          6.455300
Name: uso_limite, dtype: float64

O **uso médio do limite** é de cerca de **42%**. O mínimo é negativo (`-0,62`) porque algumas faturas têm saldo credor, e o máximo é **6,46**: um cliente com a fatura 6 vezes maior que o limite. Duas linhas de código fizeram o que o laço fez, e o pandas conta em todas as linhas sem você escrever o `for`.

**Pergunta 2: quais clientes mais estouraram o limite?**

*Com pandas:* ordenar pelo uso, do maior para o menor, e olhar os primeiros.

In [35]:
# Esta célula ordena pelo uso do limite e mostra os cinco clientes com maior uso.
maiores_usos = clientes.sort_values("uso_limite", ascending=False)
maiores_usos[["id_cliente", "limite_credito", "fatura_set", "uso_limite"]].head()

,id_cliente,limite_credito,fatura_set,uso_limite
3796,3797,20000.0,129106.0,6.4553
7508,7509,10000.0,53095.0,5.3095
20635,20636,20000.0,82812.0,4.1406
16702,16703,30000.0,118587.0,3.9529
13665,13666,20000.0,74510.0,3.7255


O cliente **3797** tem limite de 20.000 e fatura de 129.106: uso de **6,46** vezes. São casos para a equipe de risco olhar, e não para oferecer mais limite.

**Pergunta 3: como simular um aumento de 20% no limite de todos?**

In [36]:
# Esta célula cria a coluna limite_novo com aumento de 20% e compara as médias.
clientes["limite_novo"] = clientes["limite_credito"] * 1.20
clientes[["limite_credito", "limite_novo"]].mean()

limite_credito    167484.322667
limite_novo       200981.187200
dtype: float64

A média do limite sobe de **167.484** para **200.981**, os mesmos 20% de aumento que conferimos na tabela pequena.

✅ **O que o caso mostrou:** criar colunas novas com contas entre colunas, ordenar com `sort_values` e simular cenários sem laço. Aqui o pandas ganha muito: o laço vira uma linha.

### 👥 Caso 3: RH, folha de pagamento

O RH tem uma tabela pequena de colaboradores e quer consultas rápidas. Os dados são inventados, só para treinar. Cada linha é um colaborador, identificado pelo nome.

| Coluna | O que é | Unidade |
|---|---|---|
| `cargo` | Cargo do colaborador | texto |
| `setor` | Área em que trabalha | texto |
| `salario` | Salário mensal | R$ |
| `anos_casa` | Tempo de empresa | anos |

In [37]:
# Esta célula cria a tabela da folha de pagamento com cinco colaboradores.
folha = pd.DataFrame({
    "colaborador": ["Marta", "Paulo", "Renata", "Sérgio", "Tânia"],
    "cargo": ["Analista", "Gerente", "Analista", "Assistente", "Coordenadora"],
    "setor": ["Financeiro", "Comercial", "RH", "Comercial", "Financeiro"],
    "salario": [6500.0, 12000.0, 6800.0, 3500.0, 9000.0],
    "anos_casa": [4, 10, 6, 1, 8],
}).set_index("colaborador")
folha

,cargo,setor,salario,anos_casa
colaborador,,,,
Marta,Analista,Financeiro,6500.0,4
Paulo,Gerente,Comercial,12000.0,10
Renata,Analista,RH,6800.0,6
Sérgio,Assistente,Comercial,3500.0,1
Tânia,Coordenadora,Financeiro,9000.0,8


**Pergunta 1: qual o salário da Renata?** (pelo nome)

In [38]:
# Esta célula consulta o salário da Renata pelo rótulo.
folha.loc["Renata", "salario"]

np.float64(6800.0)

**Pergunta 2: quem é o terceiro colaborador da lista?** (pela posição)

In [39]:
# Esta célula consulta a terceira linha da tabela pela posição.
folha.iloc[2]

cargo        Analista
setor              RH
salario        6800.0
anos_casa           6
Name: Renata, dtype: object

A posição `2` é a **terceira** linha (a contagem começa no zero): a Renata, que é analista do RH, com 6 anos de casa. Chegamos à mesma pessoa pelo nome e pela posição.

**Pergunta 3: como montar uma lista só com setor e salário, para o gestor?**

In [40]:
# Esta célula seleciona só as colunas setor e salario da folha.
folha[["setor", "salario"]]

,setor,salario
colaborador,,
Marta,Financeiro,6500.0
Paulo,Comercial,12000.0
Renata,RH,6800.0
Sérgio,Comercial,3500.0
Tânia,Financeiro,9000.0


**Pergunta 4: qual o impacto de um reajuste de 5%?**

In [41]:
# Esta célula calcula o salário reajustado em 5% e compara o total da folha antes e depois.
folha["salario_reajustado"] = folha["salario"] * 1.05
print("Folha atual:", folha["salario"].sum())
print("Folha reajustada:", folha["salario_reajustado"].sum())

Folha atual: 37800.0
Folha reajustada: 39690.0


A folha vai de **R$ 37.800** para **R$ 39.690**, ou seja, **R$ 1.890** a mais por mês. Conferindo: 5% de 37.800 = 1.890.

✅ **O que o caso mostrou:** o mesmo vocabulário (`loc`, `iloc`, colchetes, coluna nova) vale para a folha de pagamento ou para os clientes do banco.

## 🔎 Filtros: respondendo "só os clientes que..."

Filtrar é a pergunta mais comum do gestor: *"só os clientes com tal característica"*. O pandas faz isso em dois passos, e vale entender cada um em pequeno antes dos casos.

### Passo 1: a comparação devolve uma "máscara"

Quando comparo uma coluna com um valor, o pandas responde `True` ou `False` **para cada linha**. Esse resultado se chama **máscara**.

In [42]:
# Esta célula cria a máscara de quem tem mais de 40 anos na tabela de exemplo.
mascara_idade = pessoas["idade"] > 40
mascara_idade

nome
Ana      False
Bruno     True
Carla    False
Diego     True
Name: idade, dtype: bool

Bruno (52) e Diego (61) são `True`, e Ana (35) e Carla (28) são `False`. É como marcar com um "x" quem passa no critério.

### Passo 2: passar a máscara entre colchetes

O `DataFrame` devolve só as linhas marcadas com `True`.

In [43]:
# Esta célula usa a máscara para ficar só com os clientes com mais de 40 anos.
pessoas[mascara_idade]

,idade,limite,limite_novo
nome,,,
Bruno,52,8500.0,10200.0
Diego,61,5000.0,6000.0


Vieram **Bruno e Diego**. Dá para escrever tudo numa linha só, `pessoas[pessoas["idade"] > 40]`, mas separar em dois passos ajuda a entender.

### Contar: o `True` vale 1

Como `True` vale 1 e `False` vale 0, **somar uma máscara conta quantos atendem** ao critério. E a **média** de uma máscara é a **proporção**.

In [44]:
# Esta célula conta quantos clientes têm mais de 40 anos e calcula a proporção.
print("Quantos:", mascara_idade.sum())
print("Proporção:", mascara_idade.mean())

Quantos: 2
Proporção: 0.5


São **2 de 4** clientes, ou seja, **50%**. Esse truque (`sum` e `mean` da máscara) responde "quantos?" e "qual a porcentagem?" sem precisar nem olhar a tabela.

### Combinar condições: `&` (e), `|` (ou), `~` (não)

| Operador | Significa | Exemplo |
|---|---|---|
| `&` | **e**: as duas condições ao mesmo tempo | idade > 30 **e** limite > 10.000 |
| `\|` | **ou**: pelo menos uma | idade > 50 **ou** limite > 15.000 |
| `~` | **não**: inverte a máscara | **não** tem mais de 40 anos |

Cada condição vai **entre parênteses**.

In [45]:
# Esta célula filtra quem tem idade acima de 30 E limite acima de 10 mil.
pessoas[(pessoas["idade"] > 30) & (pessoas["limite"] > 10000)]

,idade,limite,limite_novo
nome,,,
Ana,35,12000.0,14400.0


In [46]:
# Esta célula filtra quem tem idade acima de 50 OU limite acima de 15 mil.
pessoas[(pessoas["idade"] > 50) | (pessoas["limite"] > 15000)]

,idade,limite,limite_novo
nome,,,
Bruno,52,8500.0,10200.0
Carla,28,20000.0,24000.0
Diego,61,5000.0,6000.0


Conferindo na mão: no "e", só a **Ana** (35 anos, 12.000) passa nas duas. No "ou", entram **Bruno** (52 anos), **Carla** (limite 20.000) e **Diego** (61 anos). A Ana fica de fora, porque não passa em nenhuma das duas.

### `isin`: "está em uma lista"

Para perguntar "o setor é Financeiro **ou** RH", em vez de repetir vários `|`, o `isin` testa se o valor está numa lista. Vou usar a tabela de colaboradores.

In [47]:
# Esta célula filtra os colaboradores dos setores Financeiro e RH.
folha[folha["setor"].isin(["Financeiro", "RH"])]

,cargo,setor,salario,anos_casa,salario_reajustado
colaborador,,,,,
Marta,Analista,Financeiro,6500.0,4,6825.0
Renata,Analista,RH,6800.0,6,7140.0
Tânia,Coordenadora,Financeiro,9000.0,8,9450.0


Vieram **Marta, Renata e Tânia**, as três dos setores da lista.

### Dois erros clássicos

Quem vem de Python tem o hábito de escrever `and`, e quem esquece dos parênteses também tropeça. **Vou provocar o primeiro erro de propósito:** usar `and` no lugar de `&`.

In [48]:
# Esta célula usa a palavra and no lugar do operador &, o que gera erro.
pessoas[(pessoas["idade"] > 30) and (pessoas["limite"] > 10000)]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

**Como ler:** `ValueError: The truth value of a Series is ambiguous`. Traduzindo: "o `and` quer um único verdadeiro ou falso, mas você me deu uma coluna inteira de verdadeiros e falsos". Para o pandas, use `&`, `|` e `~`.

**Vou provocar o segundo erro de propósito:** esquecer os parênteses.

In [49]:
# Esta célula esquece os parênteses em volta das condições, o que gera erro.
pessoas[pessoas["idade"] > 40 & pessoas["limite"] < 10000]

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

**Como ler:** `TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]`. O `&` tem **prioridade** sobre o `>` e o `<`, então o Python tentou fazer `40 & pessoas["limite"]` antes de comparar. A regra: **cada condição entre parênteses**.

## 🛠️ Casos resolvidos: filtros

### 💰 Caso 4: Financeiro, o cliente de risco

Vamos responder à terceira pergunta do gestor, lá do início da aula, e mais duas.

**Pergunta 1: quantos clientes têm mais de 60 anos e limite acima de 200 mil?**

*Com lista:* um laço com dois `if`.

In [50]:
# Esta célula conta com um laço os clientes com mais de 60 anos e limite acima de 200 mil.
contador = 0
for idade, limite in zip(clientes["idade"], clientes["limite_credito"]):
    if idade > 60 and limite > 200000:
        contador += 1
contador

110

*Com pandas:* duas condições com `&`, e `sum` para contar.

In [51]:
# Esta célula conta os mesmos clientes com uma máscara.
mascara_idosos_limite_alto = (clientes["idade"] > 60) & (clientes["limite_credito"] > 200000)
mascara_idosos_limite_alto.sum()

np.int64(110)

São **110 clientes**, e os dois jeitos dão o mesmo número. A diferença é que a versão do pandas é uma linha, que a gente lê quase como a pergunta do gestor.

**Pergunta 2: clientes com mais de 60 anos são mais inadimplentes que a média?**

A coluna `inadimplente` tem 0 e 1, então a **média** dela dentro de um grupo é a taxa de inadimplência desse grupo.

In [52]:
# Esta célula compara a taxa de inadimplência dos clientes com mais de 60 anos com a geral.
mais_de_60 = clientes[clientes["idade"] > 60]
print("Clientes com mais de 60:", len(mais_de_60))
print("Taxa de inadimplência (mais de 60):", mais_de_60["inadimplente"].mean())
print("Taxa de inadimplência (geral):", clientes["inadimplente"].mean())

Clientes com mais de 60: 272
Taxa de inadimplência (mais de 60): 0.26838235294117646
Taxa de inadimplência (geral): 0.2212


São **272 clientes** com mais de 60 anos, com taxa de **26,8%**, contra **22,1%** da base inteira. Um pouco acima da média, mas com poucos clientes (menos de 1% da base), então vale cautela antes de concluir qualquer coisa.

**Pergunta 3: o atraso do mês anterior ajuda a prever a inadimplência?**

Vou comparar a taxa entre quem estava com **2 meses ou mais de atraso** em setembro e quem estava **sem atraso** (`atraso_set` zero ou menor).

In [53]:
# Esta célula compara a taxa de inadimplência de quem tinha atraso grave com quem não tinha atraso.
atraso_grave = clientes[clientes["atraso_set"] >= 2]
sem_atraso = clientes[clientes["atraso_set"] <= 0]
print("Atraso de 2 meses ou mais:", len(atraso_grave), "clientes, taxa", atraso_grave["inadimplente"].mean())
print("Sem atraso:", len(sem_atraso), "clientes, taxa", sem_atraso["inadimplente"].mean())

Atraso de 2 meses ou mais: 3130 clientes, taxa 0.6955271565495208
Sem atraso: 23182 clientes, taxa 0.13834009145026313


Entre os **3.130** clientes com atraso grave, a inadimplência foi de **69,6%**. Entre os **23.182** sem atraso, foi de **13,8%**: uma diferença de **5 vezes**. O atraso recente é um sinal forte, e é exatamente o tipo de resposta que o gestor de risco quer.

✅ **O que o caso mostrou:** máscara com `&`, contar com `sum`, e a **média da coluna 0/1** como taxa de um grupo.

### 🛒 Caso 5: Comercial, o público da campanha

A campanha de aumento de limite tem regras: o cliente precisa **usar mais de 80% do limite**, **estar sem atraso**, ter **pós-graduação ou universidade** (`escolaridade` 1 ou 2) e ter **entre 25 e 40 anos**.

**Pergunta 1: quantos clientes entram na campanha?**

Primeiro o `isin` para a escolaridade, depois todas as condições juntas. A coluna `uso_limite` foi criada na seção de seleção.

In [54]:
# Esta célula monta o público da campanha com quatro condições e conta quantos clientes entram.
publico_campanha = clientes[
    (clientes["uso_limite"] > 0.8)
    & (clientes["atraso_set"] <= 0)
    & (clientes["escolaridade"].isin([1, 2]))
    & (clientes["idade"] >= 25)
    & (clientes["idade"] <= 40)
]
len(publico_campanha)

2797

Entram **2.797 clientes**, cerca de **9%** da base. Repare que dividi a condição em várias linhas, uma por critério: é só uma questão de leitura, e o resultado é o mesmo.

**Pergunta 2: qual o limite médio desse público, e como ele se comporta na inadimplência?**

In [55]:
# Esta célula calcula o limite médio e a taxa de inadimplência do público da campanha.
print("Limite médio do público:", publico_campanha["limite_credito"].mean())
print("Taxa de inadimplência do público:", publico_campanha["inadimplente"].mean())

Limite médio do público: 126346.08509116911
Taxa de inadimplência do público: 0.15766893099749732


O limite médio do público é de **NT$ 126.346**, **menor** que o da base (167.484), e a inadimplência é de **15,8%**, **abaixo** dos 22,1% gerais. São clientes que usam muito o cartão e pagam em dia: um bom público para a oferta.

**Pergunta 3: e os clientes que usam mais de 80% do limite e estão sem atraso, sem os outros critérios?**

In [56]:
# Esta célula conta os clientes com uso acima de 80% e sem atraso, sem filtrar idade nem escolaridade.
usam_muito = clientes[(clientes["uso_limite"] > 0.8) & (clientes["atraso_set"] <= 0)]
len(usam_muito)

5839

São **5.839** clientes. Os critérios de idade e escolaridade reduziram o público de 5.839 para 2.797: **cada filtro a mais afina o público**, e o negócio decide até onde vale afinar.

✅ **O que o caso mostrou:** vários critérios com `&`, o `isin` para uma lista de valores e a comparação de grupos pela média.

### 👥 Caso 6: RH, consultas na folha

Voltando à tabela de 5 colaboradores, `folha`.

**Pergunta 1: quem ganha mais de R$ 6.000 e tem mais de 5 anos de casa?**

In [57]:
# Esta célula filtra colaboradores com salário acima de 6 mil e mais de 5 anos de casa.
folha[(folha["salario"] > 6000) & (folha["anos_casa"] > 5)]

,cargo,setor,salario,anos_casa,salario_reajustado
colaborador,,,,,
Paulo,Gerente,Comercial,12000.0,10,12600.0
Renata,Analista,RH,6800.0,6,7140.0
Tânia,Coordenadora,Financeiro,9000.0,8,9450.0


Vieram **Paulo, Renata e Tânia**. A Marta tem salário alto (6.500), mas só 4 anos de casa.

**Pergunta 2: quem NÃO é do setor Comercial?**

O `~` inverte a máscara.

In [58]:
# Esta célula filtra os colaboradores que não são do setor Comercial.
folha[~(folha["setor"] == "Comercial")]

,cargo,setor,salario,anos_casa,salario_reajustado
colaborador,,,,,
Marta,Analista,Financeiro,6500.0,4,6825.0
Renata,Analista,RH,6800.0,6,7140.0
Tânia,Coordenadora,Financeiro,9000.0,8,9450.0


Vieram **Marta, Renata e Tânia**: tiramos Paulo e Sérgio, que são do Comercial.

**Pergunta 3: qual a folha média do setor Financeiro?**

In [59]:
# Esta célula calcula o salário médio dos colaboradores do setor Financeiro.
folha[folha["setor"] == "Financeiro"]["salario"].mean()

np.float64(7750.0)

Marta (6.500) e Tânia (9.000) fazem **R$ 7.750** de média. Conferindo: (6.500 + 9.000) ÷ 2 = 7.750.

✅ **O que o caso mostrou:** o mesmo filtro serve para clientes do banco e para colaboradores; e `~` e `==` completam o vocabulário.

## 🧠 Questões para pensar

### ❓ Questão 1

Qual a diferença entre `loc` e `iloc`?

*Pense primeiro. A resposta está logo abaixo, escondida:*

<details>
<summary>👀 Ver a resposta</summary>

O `loc` escolhe pelos **nomes** (rótulos) das linhas e colunas e, nas fatias, **inclui o último**. O `iloc` escolhe pela **posição** (contando do zero) e **não inclui** o último.

</details>

### ❓ Questão 2

Qual a diferença entre `clientes["idade"]` e `clientes[["idade"]]`?

<details>
<summary>👀 Ver a resposta</summary>

Com colchete simples, o resultado é uma **`Series`** (uma coluna). Com colchete duplo, é um **`DataFrame`** com uma coluna só. O colchete duplo é o que se usa para escolher uma ou mais colunas e continuar com uma tabela.

</details>

### ❓ Questão 3

Para que serve a expressão `(clientes["idade"] > 60).sum()`?

<details>
<summary>👀 Ver a resposta</summary>

Ela **conta quantos clientes** têm mais de 60 anos. A comparação gera a máscara de `True` e `False`, e o `sum` soma os `True`.

</details>

### ❓ Questão 4

Por que `clientes[clientes["idade"] > 30 & clientes["limite_credito"] > 1000]` dá erro?

<details>
<summary>👀 Ver a resposta</summary>

Porque faltam os **parênteses** em volta de cada condição. O `&` tem prioridade sobre `>`, e o Python tenta calcular `30 & clientes["limite_credito"]` primeiro. O certo é `(clientes["idade"] > 30) & (clientes["limite_credito"] > 1000)`.

</details>

### ❓ Questão 5

Como saber a **taxa de inadimplência** de um grupo de clientes?

<details>
<summary>👀 Ver a resposta</summary>

Filtre o grupo e tire a **média** da coluna `inadimplente`, que só tem 0 e 1. A média de uma coluna 0/1 é a proporção de uns, ou seja, a taxa.

</details>

### ❓ Questão 6

Quando faz mais sentido usar `isin` em vez de vários `|`?

<details>
<summary>👀 Ver a resposta</summary>

Quando você quer saber se o valor está numa **lista**. `setor.isin(["RH", "Financeiro", "Comercial"])` é bem mais curto e legível do que três comparações ligadas por `|`.

</details>

## 🧾 Resumo

### Comandos

| Comando | Para que serve |
|---|---|
| `pd.read_csv(caminho)` | Lê um arquivo CSV e devolve um `DataFrame` |
| `df.head()` / `df.shape` | Primeiras linhas / número de linhas e colunas |
| `df.info()` / `df.describe()` | Tipos e valores nulos / resumo estatístico |
| `df["col"]` | Uma coluna (uma `Series`) |
| `df[["a", "b"]]` | Várias colunas (um `DataFrame`) |
| `df.loc[linha, coluna]` | Seleção por **nome**; fatia inclui o último |
| `df.iloc[linha, coluna]` | Seleção por **posição**; fatia não inclui o último |
| `df["nova"] = ...` | Cria uma coluna nova |
| `df.sort_values("col", ascending=False)` | Ordena pelo valor da coluna |
| `df[df["col"] > valor]` | Filtra linhas por uma condição |
| `(cond1) & (cond2)` / `(cond1) \| (cond2)` / `~(cond)` | E, ou, não (condições entre parênteses) |
| `df["col"].isin([a, b])` | Valor está numa lista |
| `mascara.sum()` / `mascara.mean()` | Quantos atendem / qual a proporção |

### Erros e como lê-los

| Mensagem | Causa provável |
|---|---|
| `KeyError: 'Idade'` | Nome de coluna escrito errado (maiúscula, acento, espaço) |
| `ValueError: The truth value of a Series is ambiguous` | Usou `and`, `or` ou `not` em vez de `&`, `\|` e `~` |
| `TypeError: Cannot perform 'rand_' ...` | Esqueceu os parênteses em volta das condições |
| `UFuncTypeError ... add.reduce` | Tentou somar ou tirar média de uma coluna que é texto |

## 🏋️ Exercícios para casa

Todos usam a tabela `clientes`. Escreva sua resposta na célula vazia e compare com a resposta recolhida.

### Exercício 1: contar

Quantos clientes têm menos de 25 anos?

In [60]:
# sua resposta aqui


In [61]:
# Resposta
(clientes["idade"] < 25).sum()


np.int64(2685)

### Exercício 2: duas condições

Quantas mulheres (`sexo` igual a 2) têm limite acima de 200 mil?

In [62]:
# sua resposta aqui


In [63]:
# Resposta
((clientes["sexo"] == 2) & (clientes["limite_credito"] > 200000)).sum()


np.int64(5927)

### Exercício 3: lista de valores

Entre os clientes com pós-graduação ou universidade (`escolaridade` 1 ou 2) e **50 anos ou mais**, quantos são e qual o limite médio?

*Dica: `isin([1, 2])` testa se o valor está na lista.*

In [64]:
# sua resposta aqui


In [65]:
# Resposta
grupo = clientes[clientes["escolaridade"].isin([1, 2]) & (clientes["idade"] >= 50)]
print("Quantos:", len(grupo))
print("Limite médio:", grupo["limite_credito"].mean())


Quantos: 1605
Limite médio: 195059.19003115266


### Exercício 4: taxa de um grupo

Qual a taxa de inadimplência dos clientes com **atraso de 2 meses ou mais em setembro e fatura maior que o limite** (`uso_limite` maior que 1)? Compare com a taxa geral.

In [66]:
# sua resposta aqui


In [67]:
# Resposta
alto_risco = clientes[(clientes["atraso_set"] >= 2) & (clientes["uso_limite"] > 1)]
print("Clientes:", len(alto_risco))
print("Taxa do grupo:", alto_risco["inadimplente"].mean())
print("Taxa geral:", clientes["inadimplente"].mean())


Clientes: 443
Taxa do grupo: 0.6343115124153499
Taxa geral: 0.2212


### 🚀 Desafio: o resumo para a diretoria

O gestor pediu uma conclusão sobre o **perfil de quem fica inadimplente**. Use o que viu hoje para responder:

1. Qual a taxa de inadimplência dos clientes com limite **abaixo de 50 mil** e dos com limite **acima de 200 mil**?
2. A taxa muda entre **homens** (`sexo` 1) e **mulheres** (`sexo` 2)?
3. Quantos clientes **aparecem com fatura negativa** (saldo credor) em setembro?

Depois, escreva em **2 ou 3 frases** o que você diria à diretoria.

In [68]:
# sua resposta aqui


In [69]:
# Resposta
limite_baixo = clientes[clientes["limite_credito"] < 50000]
limite_alto = clientes[clientes["limite_credito"] > 200000]
print("Limite abaixo de 50 mil:", len(limite_baixo), "clientes, taxa", limite_baixo["inadimplente"].mean())
print("Limite acima de 200 mil:", len(limite_alto), "clientes, taxa", limite_alto["inadimplente"].mean())

homens = clientes[clientes["sexo"] == 1]
mulheres = clientes[clientes["sexo"] == 2]
print("Homens: taxa", homens["inadimplente"].mean())
print("Mulheres: taxa", mulheres["inadimplente"].mean())

print("Fatura negativa em setembro:", (clientes["fatura_set"] < 0).sum())


Limite abaixo de 50 mil: 4311 clientes, taxa 0.3607051728137323
Limite acima de 200 mil: 9622 clientes, taxa 0.14726668052379963
Homens: taxa 0.2416722745625841
Mulheres: taxa 0.20776280918727916
Fatura negativa em setembro: 590


## 🚀 Próxima aula

Hoje aprendemos a **olhar, escolher e filtrar** uma tabela. Na próxima aula vamos **limpar** os dados (valores faltantes, duplicados, tipos errados) e **resumir** por grupo, com `groupby` e `pivot_table`: "qual a taxa de inadimplência **por faixa de idade**?" em uma linha só.

Deixei no **notebook complementar** os filtros que ficaram de fora (`query`, `between`, `str.contains`), a prática de Git passo a passo e mais exercícios por área.

☕ Bom descanso, e até quarta!